In [ ]:
import tempfile
import os
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import xarray as xr
from IPython.display import display, Markdown, HTML
import seaborn as sns

import oq_wrapper as oqw
import nn_gmm as nng
import ml_tools as mlt

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"

logging.basicConfig(
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    # level=logging.INFO
    level=logging.INFO
)

PLOT_IMS = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]

plt.rcParams['savefig.dpi'] = 300  

In [ ]:
result_dir = "/Users/claudy/dev/work/data/nn_gmm/results/1011_1615_cv_test_25Epochs_3E4Folds_1000Sites"

In [ ]:
# Injected Parameters
result_dir = "/home/claudy/dev/work/data/nn_gmm/results/1010_1404_cv_test_25Epochs_3E4Folds"


In [ ]:
result_dir = Path(result_dir)
# empdb_ffp = Path(os.environ["wdata"]) / "nn_gmm/20250616_CS200m_empdb.db"
empdb_ffp = Path(os.environ["wdata"]) / "nn_gmm/20251013_CS200m_empdb.duckdb"
nzgmbd_ffp = Path(os.environ["wdata"]) / "gm_datasets/nz_gmdb/v4.3_final/custom/mod_ground_motion_im_table_rotd50_flat.csv"
obs_emp_gm_params_ffp = Path(os.environ["wdata"]) / "nn_gmm/obs_emp_gm_params/emp_gm_params.parquet"

In [ ]:
cv_dirs = [cur_dir for cur_dir in result_dir.iterdir() if cur_dir.is_dir() and cur_dir.name.startswith("cv_")]

## Inputs

In [ ]:
display(Markdown(f"**Result directory**: `{result_dir}`\n\n**Empirical database**: `{empdb_ffp}`"))

In [ ]:
run_config = nng.RunConfig.from_yaml(result_dir / "run_config.yaml")
val_pred_df = pd.read_parquet(result_dir / "val_results.parquet").astype(
    {cur_key: "float32" for cur_key in list(run_config.pred_mean_keys) + list(run_config.pred_std_keys)}).sort_index()
metrics_da = xr.open_dataarray(result_dir / "metrics.nc")

### Run Config

In [ ]:
run_config

In [ ]:
val_record_int_ids = val_pred_df.index.values.astype(int)

with nng.DuckIMDB(run_config.imdb_ffp, readonly=True) as imdb:
    event_df = imdb.get_event_df()
    site_df = imdb.get_site_df()
    val_sim_df = imdb.get_im_data(run_config.ims, val_record_int_ids).sort_index()
    val_record_info_df = imdb.get_record_info_df(record_int_ids=val_record_int_ids).sort_index()
    val_site_event_df = imdb.get_site_event_df()

val_sim_df["event_int_id"] = val_record_info_df.loc[val_sim_df.index, "event_int_id"].values
val_sim_df["site_int_id"] = val_record_info_df.loc[val_sim_df.index, "site_int_id"].values
val_sim_df["site_event_int_id"] = nng.utils.get_site_event_int_id(val_sim_df.site_int_id, val_sim_df.event_int_id)
val_sim_df["magnitude"] = event_df.loc[val_sim_df["event_int_id"], "magnitude"].values
val_sim_df["rrup"] = val_site_event_df.loc[val_sim_df["site_event_int_id"], "rrup"].values
val_sim_df["vs30"] = site_df.loc[val_sim_df["site_int_id"], "vs30"].values

val_pred_df["site_int_id"] = val_record_info_df.loc[val_pred_df.index, "site_int_id"].values
assert val_pred_df.index.equals(val_sim_df.index)

In [ ]:
res_df, *_ = nng.analysis.get_nn_sim_residuals(result_dir, pred_df=val_pred_df, sim_df=val_sim_df, record_info_df=val_record_info_df)
assert res_df.index.equals(val_sim_df.index)
res_df["cv_iter"] = val_pred_df["cv_iter"]
res_df["magnitude"] = val_sim_df["magnitude"]
res_df["rrup"] = val_sim_df["rrup"]
res_df["vs30"] = val_sim_df["vs30"]

In [ ]:
emp_res_df = nng.analysis.get_emp_sim_residuals(empdb_ffp, sim_df=val_sim_df)
assert emp_res_df.index.equals(val_sim_df.index)
emp_res_df["magnitude"] = val_sim_df["magnitude"]
emp_res_df["rrup"] = val_sim_df["rrup"]
emp_res_df["vs30"] = val_sim_df["vs30"]

In [ ]:
cv_res_dfs = {cv_key: res_df.loc[res_df.cv_iter == cv_key] for cv_key in res_df.cv_iter.unique()}

## Metrics

In [ ]:
metric_keys = ["w_loss_hist","loss_hist"]

In [ ]:
for cur_metric in metric_keys:
    cur_train_df = metrics_da.loc[:, :, cur_metric + "_train"].to_pandas()
    cur_val_df = metrics_da.loc[:, :, cur_metric + "_val"].to_pandas()

    display(Markdown(f"### {cur_metric.replace('_', ' ').replace('hist', 'History').title()}"))
    display(Markdown("#### Training Data"))
    display(HTML(cur_train_df.T.describe().to_html()))

    display(Markdown("#### Validation Data"))
    display(HTML(cur_val_df.T.describe().to_html()))

In [ ]:
metric_limits = {
    "w_loss_hist": (-2.0, -0.5),
    "loss_hist": (-1.0, -0.5),
    "mse_hist": (0, 0.2),
    "mean_sigma_hist": (0, 0.5)
}

### CV-Metric Comparison

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(metric_keys) * 2, 2, -1, (8, 6))

for ix, cur_metric_key in enumerate(metric_keys):
	ax1, ax2 = axs[ix * 2], axs[ix * 2 + 1]
    
    # Get current metric values
	cur_train_metric_df = metrics_da.loc[:, :, cur_metric_key + "_train"].to_pandas().T
	cur_val_metric_df = metrics_da.loc[:, :, cur_metric_key + "_val"].to_pandas().T
    
    # Compute mean and std
	cur_mean = cur_train_metric_df.mean(axis=1)
	cur_std = cur_train_metric_df.std(axis=1)
	
	cur_val_mean = cur_val_metric_df.mean(axis=1)
	cur_val_std = cur_val_metric_df.std(axis=1)

	y_min, y_max = metric_limits[cur_metric_key]	

    # Plot
	ax1.plot(cur_train_metric_df.index.values, cur_train_metric_df, c="gray", linewidth=1.0, linestyle="--")
	ax1.plot(cur_mean.index.values, cur_mean, c="black", linewidth=2.0)
	ax1.set_ylim(y_min, y_max)
	ax1.set_title(f"Training {cur_metric_key}")
	ax1.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    
	ax2.plot(cur_val_metric_df.index.values, cur_val_metric_df, c="gray", linewidth=1.0, linestyle="--")
	ax2.plot(cur_val_mean.index.values, cur_val_mean, c="black", linewidth=2.0)
	ax2.set_ylim(y_min, y_max)
	ax2.set_title(f"Validation {cur_metric_key}")
	ax2.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    
fig.tight_layout()

### Comparison of median metrics

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(metric_keys), 1, -1, (10, 6))

for cur_ax, cur_metric_key in zip(axs, metric_keys):
    # Get current metric values
	cur_train_metric_df = metrics_da.loc[:, :, cur_metric_key + "_train"].to_pandas().T
	cur_val_metric_df = metrics_da.loc[:, :, cur_metric_key + "_val"].to_pandas().T
    
    # Get the median and 16th/84th percentiles
	train_median = cur_train_metric_df.median(axis=1)
	train_16, train_84 = np.percentile(cur_train_metric_df, [16, 84], axis=1)
	
	val_median = cur_val_metric_df.median(axis=1)
	val_16, val_84 = np.percentile(cur_val_metric_df, [16, 84], axis=1)
    
	# Get y-axis limits
	y_min, y_max = metric_limits[cur_metric_key]

	# Plot
	cur_ax.plot(cur_train_metric_df.index.values, train_median, label="Training - Median", c="tab:blue")
	cur_ax.fill_between(cur_train_metric_df.index.values, train_16, train_84, alpha=0.5, color="tab:blue")
	cur_ax.plot(cur_train_metric_df.index.values, train_16, label="Training - 16th & 84th", linestyle="--", c="tab:blue", linewidth=1.0)
	cur_ax.plot(cur_train_metric_df.index.values, train_84, linestyle="--", c="tab:blue", linewidth=1.0)

	
	cur_ax.plot(cur_val_metric_df.index.values, val_median, label="Validation - Median", c="tab:orange")
	cur_ax.fill_between(cur_val_metric_df.index.values, val_16, val_84, alpha=0.5, color="tab:orange")
	cur_ax.plot(cur_val_metric_df.index.values, val_16, label="Validation - 16th & 84th", linestyle="--", c="tab:orange", linewidth=1.0)
	cur_ax.plot(cur_val_metric_df.index.values, val_84, linestyle="--", c="tab:orange", linewidth=1.0)
	
	cur_ax.set_ylim(y_min, y_max)
	cur_ax.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	cur_ax.set_xlabel("Epoch")
	cur_ax.set_ylabel(f"{cur_metric_key}")
	cur_ax.set_title(f"{cur_metric_key}")
	cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	cur_ax.legend()
	
	fig.tight_layout()

# Simulation Analysis
Analysis of model performance on simulation data

## Model Bias & Residual Standard Deviation

### Total

In [ ]:
bias_ylim = (-0.8, 0.8)
std_ylim = (0.0, 0.8)

In [ ]:
total_bias_plt = (
    nng.plots.BiasStdPlot(bias_ylim=bias_ylim, std_ylim=std_ylim)
    .add_results(res_df, label="NN-GMM", color="b")
    .add_results(emp_res_df, label="Empirical GMM", color="g")
    .add_legend())

### CV

In [ ]:
cv_bias_plt = (
    nng.plots.BiasStdPlot()
    .add_nn_gmm_ind_cv_results(
        res_df,
        c="gray",
        linestyle="--",
        linewidth=1.0,
        alpha=0.75,
    )
    .add_results(res_df, label="NN-GMM", c="b")
    .add_nn_gmm_cv_band(res_df, color="b", alpha=0.2)
    .add_legend()
)

### Magnitude

In [ ]:
mag_bins = [5.5, 6.25, 7.25, 8.5]
mag_bin_labels = [f"$M_w$: {mag_bins[i]} - {mag_bins[i+1]}" for i in range(len(mag_bins) - 1)]
mag_bin_colors = ["blue", "purple", "red"]

In [ ]:
mag_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("magnitude", mag_bins, mag_bin_labels, mag_bin_colors)
    .add_results(emp_res_df, c="k", linestyle="--", linewidth=2.0, label="Empirical GMM")
    .add_results(res_df, c="k", linewidth=2.0, label="NN-GMM")
    .add_grouped_results(res_df, True)
    .add_grouped_results(emp_res_df, False, linestyle="--")
    .add_legend()
)

In [ ]:
cv_mag_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("magnitude", mag_bins, mag_bin_labels, mag_bin_colors)
    .add_results(res_df, c="k", linewidth=2.0)
    .add_nn_gmm_cv_band(res_df, color="k", alpha=0.2)
    .add_grouped_results(res_df, add_legend_entries=True)
    .add_nn_gmm_grouped_cv_band(res_df, alpha=0.2)
    .add_legend()
)

### $R_{Rup}$

In [ ]:
rrup_bins = [0, 30, 100, 300]
rrup_bin_labels = ["$R_{Rup}$ 0 - 30", "$R_{Rup}$ 30 - 100", "$R_{Rup}$ 100 - 300"]
rrup_bin_colors = ["blue", "purple", "red"]

In [ ]:
rrup_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("rrup", rrup_bins, rrup_bin_labels, rrup_bin_colors)
    .add_results(emp_res_df, c="k", linestyle="--", label="Empirical GMM", linewidth=2.0)
    .add_results(res_df, c="k", linewidth=2.0, label="NN-GMM")
    .add_grouped_results(res_df, add_legend_entries=True)
    .add_grouped_results(emp_res_df, add_legend_entries=False, linestyle="--")
    .add_legend()
)

In [ ]:
cv_rrup_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("rrup", rrup_bins, rrup_bin_labels, rrup_bin_colors)
    .add_results(res_df, c="k", linewidth=2.0)
    .add_nn_gmm_cv_band(res_df, color="k", alpha=0.2)
    .add_grouped_results(res_df, add_legend_entries=True)
    .add_nn_gmm_grouped_cv_band(res_df, alpha=0.2)
    .add_legend()
)

### $V_{S30}$

In [ ]:
vs30_bins = [0, 180, 360, 760, 1500]
vs30_bin_labels = ["$V_{S30}$ 0 - 180", "$V_{S30}$ 180 - 360", "$V_{S30}$ 360 - 760", "$V_{S30}$ 760 - 1500"]
vs30_bin_colors = ["blue", "purple", "red", "orange"]

In [ ]:
vs30_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("vs30", vs30_bins, vs30_bin_labels, vs30_bin_colors)
    .add_results(emp_res_df, c="k", linestyle="--", label="Empirical GMM", linewidth=2.0)
    .add_results(res_df, c="k", linewidth=2.0, label="NN-GMM")
    .add_grouped_results(res_df, add_legend_entries=True)
    .add_grouped_results(emp_res_df, add_legend_entries=False, linestyle="--")
    .add_legend()
)

In [ ]:
cv_vs30_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("vs30", vs30_bins, vs30_bin_labels, vs30_bin_colors)
    .add_results(res_df, c="k", linewidth=2.0)
    .add_nn_gmm_cv_band(res_df, color="k", alpha=0.2)
    .add_grouped_results(res_df, add_legend_entries=True)
    .add_nn_gmm_grouped_cv_band(res_df, alpha=0.2)
    .add_legend()
)

### Basin Performance

In [ ]:
site_df = nng.utils.add_basin_column(site_df)
val_sim_df["basin"] = site_df.loc[val_sim_df["site_int_id"], "basin"].values
res_df["basin"] = val_sim_df["basin"]
emp_res_df["basin"] = val_sim_df["basin"]
basin_labels = list(val_sim_df["basin"].unique())
basin_colors = sns.color_palette("tab10", len(basin_labels))

#### Number of Sites Per Basin

In [ ]:
basin_n_sites = site_df.loc[val_sim_df["site_int_id"].unique(), "basin"].value_counts()
Markdown(basin_n_sites.to_markdown())

#### Number Of Records Per Basin

In [ ]:
Markdown(val_sim_df.basin.value_counts().to_markdown())

#### Empirical Basin Bias & Residual Standard Deviation

In [ ]:
emp_basin_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("basin", None, basin_labels, basin_colors)
    .add_results(emp_res_df, c="k", linestyle="--", label="Empirical GMM", linewidth=2.0)
    .add_categorial_results(emp_res_df, add_legend_entries=True, linestyle="--")
    .add_legend()
)

#### NN Basin Bias & Residual Standard Deviation

In [ ]:
nn_basin_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("basin", None, basin_labels, basin_colors)
    .add_results(res_df, c="k", linewidth=2.0, label="NN-GMM")
    .add_categorial_results(res_df, add_legend_entries=True)
    .add_legend()
)

## Model Trends

### Magnitude

In [ ]:
major_line_width=2.0
minor_line_width=1.0

In [ ]:
# Define fixed inputs and limits for validation data points
fixed_inputs = dict(vs30=500,
    z1p0=0.13,
    z2p5=1,
    tect_type=nng.constants.REVERSE_TECTONIC_TYPE_MAPPING[oqw.constants.TectType.ACTIVE_SHALLOW],
    rake=10,
    dip=60,
    dtop=0,
    dbottom=15,
    hypo_depth=8.5,
    # Chch
    nztm_x=1570373.11078697,
    nztm_y=5179891.74707139
)
mag_limits = {
        "vs30": (25, 25),
        "z1p0": (0.25, 0.25),
        "rrup": (5, 5),
}

#### Short Distance ($R_{Rup} = 25$ km)

In [ ]:
short_dist_fixed_inputs = {"rrup": 25, "rjb": 25, "rx": 25, "ry": 25} | fixed_inputs

nng.plots.magnitude_trend_plot(result_dir, val_sim_df, short_dist_fixed_inputs, mag_limits, val_record_int_ids, device, cv=True, major_line_width=major_line_width, minor_line_width=minor_line_width);

#### Moderate Distance ($R_{Rup} = 75$ km)

In [ ]:
moderate_dist_fixed_inputs = {"rrup": 75, "rjb": 75, "rx": 75, "ry": 75} | fixed_inputs

nng.plots.magnitude_trend_plot(result_dir, val_sim_df, moderate_dist_fixed_inputs, mag_limits, val_record_int_ids, device, cv=True, major_line_width=major_line_width, minor_line_width=minor_line_width);

#### Long Distance ($R_{Rup} = 125$ km)

In [ ]:
long_dist_fixed_inputs = {"rrup": 125, "rjb": 125, "rx": 125, "ry": 125} | fixed_inputs

nng.plots.magnitude_trend_plot(result_dir, val_sim_df, long_dist_fixed_inputs, mag_limits, val_record_int_ids, device, cv=True, major_line_width=major_line_width, minor_line_width=minor_line_width);

### Source-To-Site Distance

In [ ]:
rrup_limits = {
        "vs30": (25, 25),
        "z1p0": (0.25, 0.25),
        "magnitude": (0.1, 0.1),
}

#### Magnitude 6

In [ ]:
moderate_mag_fixed_inputs = {"magnitude": 6.0} | fixed_inputs

nng.plots.rrup_trend_plot(result_dir, val_sim_df, moderate_mag_fixed_inputs, rrup_limits, val_record_int_ids, device, cv=True, major_line_width=major_line_width, minor_line_width=minor_line_width);

#### Magnitude 7

In [ ]:
moderate_mag_fixed_inputs = {"magnitude": 7.0} | fixed_inputs

nng.plots.rrup_trend_plot(result_dir, val_sim_df, moderate_mag_fixed_inputs, rrup_limits, val_record_int_ids, device, cv=True, major_line_width=major_line_width, minor_line_width=minor_line_width);

#### Magnitude 8

In [ ]:
moderate_mag_fixed_inputs = {"magnitude": 8.0} | fixed_inputs

nng.plots.rrup_trend_plot(result_dir, val_sim_df, moderate_mag_fixed_inputs, rrup_limits, val_record_int_ids, device, cv=True, major_line_width=major_line_width, minor_line_width=minor_line_width);

# Observations
Analysis of model performance with respect to NZGMDB

In [ ]:
obs_bias_ylim = (-2.0, 2.0)
obs_std_ylim = (0.0, 2.0)

In [ ]:
### FIX THIS
obs_data = nng.obs_data.load_obs_nzgmdb(nzgmbd_ffp, apply_mag_distance_filter=False)

In [ ]:
# Load observation predictions
obs_preds = {cur_dir.stem: pd.read_parquet(cur_dir / "obs_results.parquet") for cur_dir in cv_dirs}

In [ ]:
# Load observation prediction into xarray DataArray
obs_cols = list(run_config.pred_mean_keys) + list(run_config.pred_std_keys)
pred_values = np.stack([pred_df[obs_cols].values for cv_key, pred_df in obs_preds.items()], axis=0)

obs_pred_da = xr.DataArray(
    data=pred_values,
    dims=["cv_iter", "record_id", "pred_key"],
    coords={
        "cv_iter": list(obs_preds.keys()),
        "record_id": obs_preds[list(obs_preds.keys())[0]].index.values.astype(str),
        "pred_key": obs_cols
    }
).transpose("record_id", "pred_key", "cv_iter")

In [ ]:
# Compute observation residuals
obs_res_values = obs_pred_da.sel(pred_key=run_config.pred_mean_keys).values - np.log(obs_data.record_df.loc[obs_pred_da.record_id.values, run_config.ims].values[..., None])
obs_res_da = xr.DataArray(
    data=obs_res_values,
    dims=["record_id", "pred_key", "cv_iter"],
    coords={
        "record_id": obs_pred_da.record_id.values,
        "pred_key": run_config.ims,
        "cv_iter": obs_pred_da.cv_iter.values
    }
)

In [ ]:
# Combine into single DataFrame
obs_res_df = pd.concat([obs_res_da.sel(cv_iter=cur_cv).to_pandas().assign(cv_iter=cur_cv) for cur_cv in obs_res_da.cv_iter.values], axis=0)
obs_res_df = obs_res_df.astype({"cv_iter": "category"})
obs_res_df["magnitude"] = obs_data.record_df.loc[obs_res_df.index, nng.obs_data.ObservedData.EventColEnums.MAG].values
obs_res_df["rrup"] = obs_data.record_df.loc[obs_res_df.index, nng.obs_data.ObservedData.EventSiteColEnums.RRUP].values
obs_res_df["vs30"] = obs_data.record_df.loc[obs_res_df.index, nng.obs_data.ObservedData.SiteColEnums.VS30].values

In [ ]:
# Compute empirical GMM residuals for observations
obs_emp_gm_params_df = pd.read_parquet(obs_emp_gm_params_ffp)
im_mean_cols = [f"{cur_im}_mean" for cur_im in run_config.ims]
obs_emp_res_df = pd.DataFrame(data=obs_emp_gm_params_df.loc[obs_pred_da.record_id.values, im_mean_cols].values - np.log(obs_data.record_df.loc[obs_pred_da.record_id.values, run_config.ims].values), index=obs_pred_da.record_id.values, columns=run_config.ims)
obs_emp_res_df["magnitude"] = obs_data.record_df.loc[obs_emp_res_df.index, nng.obs_data.ObservedData.EventColEnums.MAG].values
obs_emp_res_df["rrup"] = obs_data.record_df.loc[obs_emp_res_df.index, nng.obs_data.ObservedData.EventSiteColEnums.RRUP].values
obs_emp_res_df["vs30"] = obs_data.record_df.loc[obs_emp_res_df.index, nng.obs_data.ObservedData.SiteColEnums.VS30].values

## Model Bias & Residual Standard Deviation

### Total

In [ ]:
obs_bias_std_plot = (
    nng.plots.BiasStdPlot(bias_ylim=obs_bias_ylim, std_ylim=obs_std_ylim)
    .add_nn_gmm_ind_cv_results(obs_res_df, linestyle="--", c="gray", alpha=0.75)
    .add_nn_gmm_cv_band(obs_res_df, color="b", alpha=0.2)
    .add_results(obs_res_df, label="NN-GMM", c="b", linewidth=2.0)
    .add_results(obs_emp_res_df, label="Empirical GMM", c="g", linewidth=2.0)
    .add_legend()
)

### Magnitude

In [ ]:
obs_mag_bins = [3.5, 5.5, 6.25, 7.25, 8.5]
obs_mag_bin_labels = [f"$M_w$: {obs_mag_bins[i]} - {obs_mag_bins[i+1]}" for i in range(len(obs_mag_bins) - 1)]
obs_mag_bin_colors = ["gray", "blue", "purple", "red"]

In [ ]:
mag_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("magnitude", obs_mag_bins, obs_mag_bin_labels, obs_mag_bin_colors, bias_ylim=obs_bias_ylim, std_ylim=obs_std_ylim)
    .add_results(obs_emp_res_df, c="k", linestyle="--", linewidth=2.0, label="Empirical GMM")
    .add_results(obs_res_df, c="k", linewidth=2.0, label="NN-GMM")
    .add_grouped_results(obs_res_df, True)
    .add_grouped_results(obs_emp_res_df, False, linestyle="--")
    # .add_grouped_results(emp_res_df, False, linestyle="--")
    .add_legend()
)

In [ ]:
cv_mag_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("magnitude", obs_mag_bins, obs_mag_bin_labels, obs_mag_bin_colors, bias_ylim=obs_bias_ylim, std_ylim=obs_std_ylim)
    .add_results(obs_emp_res_df, c="k", linestyle="--", linewidth=2.0, label="Empirical GMM")
    .add_results(obs_res_df, c="k", linewidth=2.0)
    .add_nn_gmm_cv_band(obs_res_df, color="k", alpha=0.2)
    .add_grouped_results(obs_res_df, add_legend_entries=True)
    .add_nn_gmm_grouped_cv_band(obs_res_df, alpha=0.2)
    .add_grouped_results(obs_emp_res_df, False, linestyle="--")
    .add_legend()
)

### $R_{rup}$

In [ ]:
obs_rrup_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("rrup", rrup_bins, rrup_bin_labels, rrup_bin_colors, bias_ylim=obs_bias_ylim, std_ylim=obs_std_ylim)
    .add_results(obs_emp_res_df, c="k", linestyle="--", label="Empirical GMM", linewidth=2.0)
    .add_results(obs_res_df, c="k", linewidth=2.0, label="NN-GMM")
    .add_grouped_results(obs_res_df, add_legend_entries=True)
    .add_grouped_results(obs_emp_res_df, add_legend_entries=False, linestyle="--")
    .add_legend()
)

In [ ]:
cv_rrup_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("rrup", rrup_bins, rrup_bin_labels, rrup_bin_colors, bias_ylim=obs_bias_ylim, std_ylim=obs_std_ylim)
    .add_results(obs_res_df, c="k", linewidth=2.0)
    .add_nn_gmm_cv_band(obs_res_df, color="k", alpha=0.2)
    .add_grouped_results(obs_res_df, add_legend_entries=True)
    .add_nn_gmm_grouped_cv_band(obs_res_df, alpha=0.2)
    .add_legend()
)

## $V_{S30}$

In [ ]:
vs30_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("vs30", vs30_bins, vs30_bin_labels, vs30_bin_colors, bias_ylim=obs_bias_ylim, std_ylim=obs_std_ylim)
    .add_results(obs_emp_res_df, c="k", linestyle="--", label="Empirical GMM", linewidth=2.0)
    .add_results(obs_res_df, c="k", linewidth=2.0, label="NN-GMM")
    .add_grouped_results(obs_res_df, add_legend_entries=True)
    .add_grouped_results(obs_emp_res_df, add_legend_entries=False, linestyle="--")
    .add_legend()
)

In [ ]:
cv_vs30_bias_std_plot = (
    nng.plots.GroupedBiasStdPlot("vs30", vs30_bins, vs30_bin_labels, vs30_bin_colors, bias_ylim=obs_bias_ylim, std_ylim=obs_std_ylim)
    .add_results(obs_res_df, c="k", linewidth=2.0)
    .add_nn_gmm_cv_band(obs_res_df, color="k", alpha=0.2)
    .add_grouped_results(obs_res_df, add_legend_entries=True)
    .add_nn_gmm_grouped_cv_band(obs_res_df, alpha=0.2)
    .add_legend()
)